# Model Evaluation & Backtesting


## Purpose
Comprehensive walk-forward backtesting 2000–2025. Calibration analysis. Error analysis on historical upsets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg


## Load Backtest Results

In [ ]:
backtest_path = cfg.project_root / "reports" / "backtest_results.csv"
if backtest_path.exists():
    results = pd.read_csv(backtest_path)
    print(results.describe())
else:
    print("Run make evaluate first")


## Accuracy Over Time

In [ ]:
Path("../reports/figures").mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.bar(results["test_season"], results["accuracy"], color="steelblue", alpha=0.8, label="Model accuracy")
ax.bar(results["test_season"], results["naive_accuracy"], color="lightgray", alpha=0.6,
       label="Naive accuracy", zorder=0)
ax.axhline(results["accuracy"].mean(), color="navy", ls="--", lw=1.5,
           label=f"Mean={results['accuracy'].mean():.3f}")
ax.set_xlabel("Test Season"); ax.set_ylabel("Accuracy")
ax.set_title("Walk-Forward Accuracy by Season")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

ax2 = axes[0, 1]
ax2.bar(results["test_season"], results["accuracy"] - results["naive_accuracy"],
        color=["steelblue" if v >= 0 else "tomato"
               for v in (results["accuracy"] - results["naive_accuracy"])],
        alpha=0.8)
ax2.axhline(0, color="black", lw=1)
ax2.set_xlabel("Test Season"); ax2.set_ylabel("Accuracy − Naive")
ax2.set_title("Model vs Naive Accuracy (Lift)")
ax2.grid(alpha=0.3, axis="y")

ax3 = axes[1, 0]
ax3.plot(results["test_season"], results["log_loss"], "o-", color="tomato", lw=2, ms=6)
ax3.axhline(results["log_loss"].mean(), color="darkred", ls="--", lw=1.5,
            label=f"Mean={results['log_loss'].mean():.3f}")
ax3.axhline(0.52, color="orange", ls=":", lw=1.5, label="Target: < 0.52")
ax3.set_xlabel("Test Season"); ax3.set_ylabel("Log-loss")
ax3.set_title("Log-Loss by Season (lower = better)")
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

ax4 = axes[1, 1]
ax4.plot(results["test_season"], results["ece"], "o-", color="mediumorchid", lw=2, ms=6)
ax4.axhline(results["ece"].mean(), color="purple", ls="--", lw=1.5,
            label=f"Mean={results['ece'].mean():.3f}")
ax4.axhline(0.05, color="green", ls=":", lw=1.5, label="Target: < 0.05")
ax4.set_xlabel("Test Season"); ax4.set_ylabel("ECE")
ax4.set_title("Expected Calibration Error by Season")
ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

plt.suptitle("Model Performance Across All Backtest Seasons", fontsize=13)
plt.tight_layout()
plt.savefig("../reports/figures/08_accuracy_over_time.png", dpi=120, bbox_inches="tight")
plt.show()

best = results.loc[results["accuracy"].idxmax()]
worst = results.loc[results["accuracy"].idxmin()]
print(f"Best season:  {int(best['test_season'])} — acc={best['accuracy']:.3f}")
print(f"Worst season: {int(worst['test_season'])} — acc={worst['accuracy']:.3f}")


## Calibration — Reliability Diagram

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from nba_predictor.evaluation.cv_strategy import playoff_season_cv_splits
from nba_predictor.models.ensemble import get_feature_cols as get_winner_feat_cols
import pickle

series_path = cfg.project_root / "data/processed/series_dataset.parquet"
df = pd.read_parquet(series_path)
feat_cols = get_winner_feat_cols(df)

# Re-generate OOF probabilities from the baseline LR for calibration analysis
all_probs, all_targets = [], []
for train_idx, test_idx in playoff_season_cv_splits(df):
    train, test = df.loc[train_idx], df.loc[test_idx]
    X_tr = train[feat_cols].fillna(0); y_tr = train["higher_seed_wins"].values
    X_te = test[feat_cols].fillna(0);  y_te = test["higher_seed_wins"].values
    scaler = StandardScaler()
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(scaler.fit_transform(X_tr), y_tr)
    probs = lr.predict_proba(scaler.transform(X_te))[:, 1]
    all_probs.extend(probs.tolist()); all_targets.extend(y_te.tolist())

all_probs = np.array(all_probs); all_targets = np.array(all_targets)

n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
frac_pos, mean_pred, bin_counts = [], [], []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (all_probs >= lo) & (all_probs < hi)
    if mask.sum() > 0:
        frac_pos.append(all_targets[mask].mean())
        mean_pred.append(all_probs[mask].mean())
        bin_counts.append(mask.sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")
ax.plot(mean_pred, frac_pos, "o-", color="steelblue", lw=2, ms=8, label="Model")
for mp, fp, n in zip(mean_pred, frac_pos, bin_counts):
    ax.text(mp, fp + 0.03, str(n), ha="center", fontsize=7, color="steelblue")
ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.15, color="steelblue")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Actual win rate")
ax.set_title("Reliability Diagram (n = sample count per bin)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.1); ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.hist(all_probs[all_targets==1], bins=20, alpha=0.6, color="steelblue",
         label="Higher seed wins", density=True)
ax2.hist(all_probs[all_targets==0], bins=20, alpha=0.6, color="tomato",
         label="Upset", density=True)
ax2.axvline(0.5, color="black", ls="--", lw=1)
ax2.set_xlabel("P(higher seed wins)"); ax2.set_title("OOF Probability Distribution")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/figures/08_calibration.png", dpi=120, bbox_inches="tight")
plt.show()

ece = np.mean([abs(fp - mp) for fp, mp in zip(frac_pos, mean_pred)])
print(f"ECE: {ece:.4f}  (target: < 0.05)")
print(f"Note: ECE > 0.05 means calibration could be improved with isotonic regression post-processing")


## Upset Analysis

In [ ]:
series_path = cfg.project_root / "data/processed/series_dataset.parquet"
df = pd.read_parquet(series_path)

upsets = df[df["higher_seed_wins"] == 0].copy()
print(f"Total upsets (2003–2025): {len(upsets)} / {len(df)} = {len(upsets)/len(df):.1%}")

print(f"\nUpsets by seed differential:")
for sd in sorted(upsets["seed_diff"].dropna().unique()):
    n_upset = (upsets["seed_diff"] == sd).sum()
    n_total = (df["seed_diff"] == sd).sum()
    print(f"  seed_diff={int(sd):2d}  upsets={n_upset}/{n_total}  ({n_upset/n_total:.1%})")

print(f"\nUpsets by series round:")
for rnd in upsets["series_round"].value_counts().index:
    n_upset = (upsets["series_round"] == rnd).sum()
    n_total = (df["series_round"] == rnd).sum()
    print(f"  {rnd:45s}  {n_upset}/{n_total}  ({n_upset/n_total:.1%})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(df[df["higher_seed_wins"]==1]["delta_NRtg"].dropna(), bins=25, alpha=0.5,
        color="steelblue", label="Favorite wins", density=True)
ax.hist(upsets["delta_NRtg"].dropna(), bins=20, alpha=0.7,
        color="tomato", label="Upset", density=True)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("delta_NRtg (higher − lower seed)")
ax.set_title("Net Rating Differential in Upsets vs Favorites")
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.hist(df[df["higher_seed_wins"]==1]["delta_VORP"].dropna(), bins=25, alpha=0.5,
         color="steelblue", label="Favorite wins", density=True)
ax2.hist(upsets["delta_VORP"].dropna(), bins=20, alpha=0.7,
         color="tomato", label="Upset", density=True)
ax2.axvline(0, color="black", lw=1)
ax2.set_xlabel("delta_VORP")
ax2.set_title("VORP Differential in Upsets")
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Anatomy of NBA Playoff Upsets (2003–2025)", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/figures/08_upset_analysis.png", dpi=120, bbox_inches="tight")
plt.show()

big_upsets = upsets.nsmallest(5, "delta_NRtg")[
    ["season", "higher_seed", "lower_seed", "seed_diff", "series_round",
     "delta_NRtg", "delta_VORP", "series_length"]]
print("\nBiggest NRtg-delta upsets (hardest for model to call):")
print(big_upsets.to_string(index=False))


## Metric Summary vs. Targets

In [ ]:
print("Target benchmarks:")
print("  Accuracy > 76%    (naive: 71%)")
print("  Log-loss < 0.52")
print("  Brier score < 0.19")
print("  ECE < 0.05")
